In [ ]:
from dotenv import load_dotenv          
load_dotenv() 

In [ ]:
# Error Handling: The SDKs throw typed exceptions that make error handling straightforward:
import anthropic

# The SDK reads ANTHROPIC_API_KEY from your environment
client = anthropic.Anthropic()

try: message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Hello, Claude!"}
    ]
)

except anthropic.AuthenticationError:
    print("Invalid API key")
except anthropic.RateLimitError:
    print("Rate limited -- back off and retry")
except anthropic.APIError as e:
    print(f"API error: {e.status_code}")

In [ ]:
# Streaming: Without streaming, your application waits for the entire response 
# before showing anything to the user. With streaming, tokens appear in real-time 
# as Claude generates them, creating a dramatically better user experience.
with client.messages.stream(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[{"role": "user", "content": "Write a poem about APIs."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

In [ ]:
# Using Tools: Claude can call external tools to get real-time information,
#  perform calculations, or interact with APIs. When Claude calls a tool, 
# you receive a structured event with the tool name and input parameters, 
# allowing your application to execute the tool and return the results back to Claude in real-time.
with client.messages.stream(
  model="claude-sonnet-4-6",
  max_tokens=1024,
  tools=[{
    "name": "get_weather",
    "description": "Get current weather for a location",
    "input_schema": {
      "type": "object",
      "properties": {
        "location": {"type": "string"},
        "units": {"type": "string", "enum": ["celsius", "fahrenheit"]}
      },
      "required": ["location"]
    }
  }],
  messages=[
    {"role": "user", "content": "What's the weather in Paris?"}
  ]
  ) as stream:
    for event in stream:
        # This captures ALL block types, including tool_use
        print(event)

In [ ]:
# Retry Strategy with Exponential Backoff
import time
import anthropic

client = anthropic.Anthropic()

def call_with_retry(messages, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=1024,
                messages=messages
            )
        except anthropic.RateLimitError:
            wait = 2 ** attempt  # 1s, 2s, 4s
            print(f"Rate limited. Retrying in {wait}s...")
            time.sleep(wait)
        except anthropic.APIStatusError as e:
            if e.status_code == 529:
                wait = 2 ** (attempt + 1)  # Longer backoff for overload
                time.sleep(wait)
            else:
                raise
    raise Exception("Max retries exceeded")

In [ ]:
# Execute function with retry logic
# pip install rich
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel

messages=[{"role": "user", "content": "Write a poem about APIs."}]
message=call_with_retry(messages, max_retries=3)
console = Console()
console.print(Panel(Markdown(message.content[0].text), title="Claude's Response", border_style="blue"))
